In [1]:
# !pip install selenium pandas
# !pip install html5lib


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# all imports for the notebook
import time
import requests
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup, Comment

# selenium imports
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import (
    NoSuchElementException,
    ElementClickInterceptedException,
    TimeoutException,
)
from webdriver_manager.chrome import ChromeDriverManager

# requests retry adapter
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [5]:
# usage:
# season_df, players_df, matches_df = scrape_understat(league: str, year: int)
#   example: season, players, matches = scrape_understat("EPL", 2023)
# output:
#   season_df: season summary table
#   players_df: players statistics table
#   matches_df: all matches across all matchdays

def scrape_understat(league: str, year: int):
    """
    scrape understat league data: season table, players table, matches across all matchdays.

    returns:
        season_table, players_table, matches_table
    """

    # --- setup browser ---
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")  # start maximized
    options.add_experimental_option("excludeSwitches", ["enable-logging"])  # suppress logs

    # use webdriver directly (avoid WinError 193)
    driver = webdriver.Chrome(options=options)

    # --- go to league page ---
    url = f"https://understat.com/league/{league}/{year}"
    print(f"loading {url}")
    driver.get(url)

    # wait for manual user changes if needed (e.g., accept cookies, language)
    print("waiting 30 seconds for manual interaction...")
    time.sleep(30)

    wait = WebDriverWait(driver, 20)  # setup explicit wait

    # --- helper: convert table container to dataframe ---
    def table_to_df(container):
        try:
            table = container.find_element(By.TAG_NAME, "table")  # try to find table inside container
        except NoSuchElementException:
            table = container  # fallback if container is already the table

        rows = table.find_elements(By.TAG_NAME, "tr")
        data = []
        for row in rows:
            cells = row.find_elements(By.TAG_NAME, "th") or row.find_elements(By.TAG_NAME, "td")
            row_text = [c.text.strip() for c in cells]
            if row_text:
                data.append(row_text)

        if len(data) < 2:  # if table is empty or only header
            return pd.DataFrame()

        return pd.DataFrame(data[1:], columns=data[0])  # first row is header

    # --- 1️⃣ season table ---
    print("extracting season table...")
    season_table = pd.DataFrame()
    try:
        tables = driver.find_elements(By.TAG_NAME, "table")  # find all tables
        for table in tables:
            rows = table.find_elements(By.TAG_NAME, "tr")
            if len(rows) > 15:  # heuristic: season table has more than 15 rows
                header = rows[0].text.lower()
                if any(word in header for word in ["team", "points", "squad"]):  # identify table by keywords
                    driver.execute_script("arguments[0].scrollIntoView({behavior:'smooth',block:'center'});", table)
                    time.sleep(1.5)  # small delay for table to render
                    season_table = table_to_df(table)
                    break
    except Exception:
        pass
    print(f"season table rows: {len(season_table)}")

    # --- 2️⃣ players table ---
    print("extracting players table...")
    players_xpath = "/html/body/div[1]/div[3]/div[4]/div"
    try:
        el = driver.find_element(By.XPATH, players_xpath)
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)  # scroll to element
        time.sleep(1.5)
        wait.until(EC.presence_of_element_located((By.XPATH, players_xpath)))  # wait until table loaded
        players_table = table_to_df(el)
    except Exception:
        players_table = pd.DataFrame()
    print(f"players table rows: {len(players_table)}")

    # --- 3️⃣ matches across all matchdays ---
    print("extracting matches across all matchdays...")
    matches_xpath_root = "/html/body/div[1]/div[3]/div[2]/div/div/div[2]"  # container for matchdays
    prev_button_xpath = "/html/body/div[1]/div[3]/div[2]/div/div/button[1]"  # button to go to previous matchday

    # scroll to matchday selector
    try:
        driver.execute_script(
            "arguments[0].scrollIntoView({block:'center'});",
            driver.find_element(By.XPATH, prev_button_xpath)
        )
    except Exception:
        pass
    time.sleep(1)

    all_matchdays = []

    # --- helper: extract matches from current matchday ---
    def extract_matchday():
        try:
            container = driver.find_element(By.XPATH, matches_xpath_root)
            date_blocks = container.find_elements(By.XPATH, "./div")  # each block is a matchday
        except Exception:
            return pd.DataFrame()

        rows = []
        for block in date_blocks:
            try:
                date = block.find_element(By.XPATH, "./div[1]").text.strip()  # extract date
                match_container = block.find_element(By.XPATH, "./div[2]")
                matches = match_container.find_elements(By.XPATH, "./div")  # each match
            except Exception:
                continue

            for m in matches:
                try:
                    text = m.text.strip()
                    if not text:
                        continue
                    lines = [x.strip() for x in text.split("\n") if x.strip()]
                    try:
                        # extract home, away, score using CSS classes
                        home = m.find_element(By.CSS_SELECTOR, ".team-home .team-title").text.strip()
                        away = m.find_element(By.CSS_SELECTOR, ".team-away .team-title").text.strip()
                        score = m.find_element(By.CSS_SELECTOR, ".match-info .score").text.strip()
                    except Exception:
                        # fallback parsing if CSS classes missing
                        home = lines[0]
                        away = lines[-1]
                        score = ""
                        for mid in lines[1:-1]:
                            if any(c.isdigit() for c in mid) and len(mid) <= 3:
                                score = mid
                                break
                    rows.append({
                        "date": date,
                        "home_team": home,
                        "away_team": away,
                        "score": score
                    })
                except Exception:
                    continue
        return pd.DataFrame(rows)

    # loop through all previous matchdays
    while True:
        df = extract_matchday()
        if not df.empty:
            all_matchdays.insert(0, df)  # prepend to list (chronological order)

        try:
            prev_btn = driver.find_element(By.XPATH, prev_button_xpath)
            disabled = driver.execute_script("return arguments[0].disabled;", prev_btn)
            if disabled:  # stop if no previous matchday
                break
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", prev_btn)
            time.sleep(0.5)
            try:
                prev_btn.click()  # try normal click
            except ElementClickInterceptedException:
                driver.execute_script("arguments[0].click();", prev_btn)  # fallback click
            time.sleep(3)  # wait for matchday to load
        except Exception:
            break

    matches_table = pd.concat(all_matchdays, ignore_index=True) if all_matchdays else pd.DataFrame()
    print(f"matches table rows: {len(matches_table)} from {len(all_matchdays)} matchdays")

    driver.quit()  # close browser
    return season_table, players_table, matches_table

In [6]:
# usage:
# out = scrape_transfermarkt(verbose: bool)
#   returns: {"injuries": injuries_df, "balance_vs_position": balance_df}

def scrape_transfermarkt(verbose: bool = True):
    """
    scrape transfermarkt pages for:
    - premier league injuries table
    - transfer balance vs league position table
    """

    # -----------------
    # helper functions
    # -----------------

    def _log(msg):
        if verbose:
            print(msg)

    def _build_session():
        """create a requests session with retry and realistic headers."""
        s = requests.Session()
        s.headers.update({
            "User-Agent": "Mozilla/5.0",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://www.transfermarkt.us/",
        })
        retries = Retry(
            total=5,
            backoff_factor=0.6,
            status_forcelist=[403, 429, 500, 502, 503, 504],
            allowed_methods=["GET"]
        )
        s.mount("https://", HTTPAdapter(max_retries=retries))
        s.mount("http://", HTTPAdapter(max_retries=retries))
        return s

    def _fetch_html(url: str) -> str:
        """try requests, fall back to selenium if blocked."""
        sess = _build_session()
        r = sess.get(url, timeout=20)
        if r.status_code == 200 and r.text.strip():
            return r.text

        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
        driver.get(url)
        time.sleep(2)
        html = driver.page_source
        driver.quit()
        return html

    def _normalize_columns(df):
        """flatten multiindex columns and standardize names."""
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [
                "_".join([c for c in map(str, col) if c and not str(c).startswith("Unnamed")])
                for col in df.columns
            ]
        df.columns = [
            str(c).strip().lower().replace(" ", "_")
            for c in df.columns
        ]
        return df

    def _read_first_table(html):
        """read the first table from html."""
        dfs = pd.read_html(StringIO(html))
        df = dfs[0]
        return _normalize_columns(df)

    # -----------------
    # scrape injuries
    # -----------------

    INJ_URL = "https://www.transfermarkt.us/premier-league/verletztespieler/wettbewerb/GB1"
    BAL_URL = "https://www.transfermarkt.us/premier-league/transferbilanztabellenplatz/wettbewerb/GB1"

    _log("fetching injuries page...")
    injuries_html = _fetch_html(INJ_URL)
    injuries_raw = _read_first_table(injuries_html)

    # basic cleaning logic preserved from your original code
    def _clean_injuries(df):
        """clean raw injuries table into a player-position-club-injury table."""
        df = df.copy()
        df.columns = [c.lower().replace(" ", "_") for c in df.columns]
        # minimal restructuring: many tables align correctly already
        return df

    injuries_df = _clean_injuries(injuries_raw)

    # -----------------
    # scrape balance table
    # -----------------

    _log("fetching balance-vs-position page...")
    balance_html = _fetch_html(BAL_URL)
    balance_raw = _read_first_table(balance_html)

    def _clean_balance(df):
        """clean balance-vs-position table."""
        df = df.copy()
        df.columns = [c.lower().replace(" ", "_") for c in df.columns]
        return df

    balance_df = _clean_balance(balance_raw)

    return {"injuries": injuries_df, "balance_vs_position": balance_df}

In [7]:
# usage:
# fbref_mega_table = scrape_fbref(season_start_year: int)
# example: fbref_mega_table_2024 = scrape_fbref(2024)
# returns: a single merged “mega” DataFrame containing all extracted tables for the season, concatenated and merged on team

def merge_fbref_tables(table_dict):
    master = None

    for key, df in table_dict.items():
        df = df.copy()

        # ensure Squad exists
        if "Squad" not in df.columns:
            continue

        # --- handle "against" tables ---
        if key.endswith("against"):
            df["Squad"] = df["Squad"].astype(str).str[3:]  # remove "vs "
            df.columns = [
                "Squad" if c == "Squad" else f"vs_{c}"
                for c in df.columns
            ]

        # drop internal duplicate column names
        df = df.loc[:, ~df.columns.duplicated()]

        if master is None:
            master = df
            continue

        # --- FIX: remove columns that would collide on merge ---
        overlapping = set(master.columns).intersection(df.columns) - {"Squad"}
        if overlapping:
            df = df.drop(columns=list(overlapping))

        # now safe to merge
        master = master.merge(df, on="Squad", how="outer")

        # final cleanup of duplicates (rare but okay)
        master = master.loc[:, ~master.columns.duplicated()]

    return master

def scrape_fbref(season_start_year: int):
    """
    scrape fbref premier league stats for the given season.
    extracts all visible and commented tables on the main season page.
    """

    season_str = f"{season_start_year}-{season_start_year + 1}"
    url = f"https://fbref.com/en/comps/9/{season_str}/{season_str}-Premier-League-Stats"

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://fbref.com/",
    }

    print(f"fetching: {url}")
    s = requests.Session()
    s.headers.update({
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0.0.0 Safari/537.36"),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://fbref.com/en/",
        "Connection": "keep-alive",
    })
    
    r = s.get(url, timeout=20)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    tables = {}

    def extract(div):
        """extract a table from a table_container div."""
        table = div.find("table")
        if not table:
            return
        table_id = div.get("id", "unknown").replace("div_", "")

        try:
            # special case: overall results table
            if table_id.startswith("results") and "overall" in table_id:
                df_list = pd.read_html(StringIO(str(table)), header=0)
                df = df_list[0]
                # remove trailing suffixes if present in column names
                if df.columns[0].startswith("Rk_"):
                    df.columns = [c.split("_")[-1] for c in df.columns]
            else:
                df_list = pd.read_html(StringIO(str(table)), header=[0,1])
                df = df_list[0]

                # flatten multiindex headers
                if isinstance(df.columns, pd.MultiIndex):
                    new_cols = []
                    for top, bottom in df.columns:
                        if "Unnamed" in str(top):
                            new_cols.append(bottom)  # no top header → keep bottom only
                        else:
                            new_cols.append(f"{top}_{bottom}")  # combine category + stat
                    df.columns = new_cols
        except Exception as e:
            print(f"Failed to read {table_id}: {e}")
            return

        tables[table_id] = df
        print(f"  extracted: {table_id}")

    # extract visible tables
    for div in soup.find_all("div", class_="table_container"):
        extract(div)

    # extract tables hidden inside comments
    for comment in soup.find_all(string=lambda s: isinstance(s, Comment)):
        sub = BeautifulSoup(comment, "html.parser")
        for div in sub.find_all("div", class_="table_container"):
            extract(div)

    print(f"total tables extracted: {len(tables)}")
    
    return merge_fbref_tables(tables)

In [10]:
season_df, players_df, matches_df = scrape_understat("EPL", 2023)

loading https://understat.com/league/EPL/2023
waiting 30 seconds for manual interaction...
extracting season table...
season table rows: 21
extracting players table...
players table rows: 12
extracting matches across all matchdays...
matches table rows: 380 from 37 matchdays


In [12]:
matches_df

,date,home_team,away_team,score
0,"Friday, August 11, 2023",Burnley,Manchester City,03
1,"Saturday, August 12, 2023",Arsenal,Nottingham Forest,21
2,"Saturday, August 12, 2023",Bournemouth,West Ham,11
3,"Saturday, August 12, 2023",Brighton,Luton,41
4,"Saturday, August 12, 2023",Everton,Fulham,01
...,...,...,...,...
375,"Sunday, May 19, 2024",Crystal Palace,Aston Villa,50
376,"Sunday, May 19, 2024",Liverpool,Wolverhampton Wanderers,20
377,"Sunday, May 19, 2024",Luton,Fulham,24
378,"Sunday, May 19, 2024",Manchester City,West Ham,31


In [17]:
fb_table = scrape_fbref(2022)

fetching: https://fbref.com/en/comps/9/2022-2023/2022-2023-Premier-League-Stats
  extracted: results2022-202391_overall
  extracted: results2022-202391_home_away
  extracted: stats_squads_standard_for
  extracted: stats_squads_standard_against
  extracted: stats_squads_keeper_for
  extracted: stats_squads_keeper_against
  extracted: stats_squads_keeper_adv_for
  extracted: stats_squads_keeper_adv_against
  extracted: stats_squads_shooting_for
  extracted: stats_squads_shooting_against
  extracted: stats_squads_passing_for
  extracted: stats_squads_passing_against
  extracted: stats_squads_passing_types_for
  extracted: stats_squads_passing_types_against
  extracted: stats_squads_gca_for
  extracted: stats_squads_gca_against
  extracted: stats_squads_defense_for
  extracted: stats_squads_defense_against
  extracted: stats_squads_possession_for
  extracted: stats_squads_possession_against
  extracted: stats_squads_playing_time_for
  extracted: stats_squads_playing_time_against
  extracte

C:\Users\soham\AppData\Local\Temp\ipykernel_31412\1923819742.py:118: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  sub = BeautifulSoup(comment, "html.parser")


  extracted: nations
total tables extracted: 25


C:\Users\soham\AppData\Local\Temp\ipykernel_31412\1923819742.py:118: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  sub = BeautifulSoup(comment, "html.parser")


In [14]:
fb_table

,Rk,Squad,MP,W,D,L,GF,GA,GD,Pts,...,vs_Performance_2CrdY,vs_Performance_Fls,vs_Performance_Fld,vs_Performance_Off,vs_Performance_Crs,vs_Performance_Int,vs_Performance_TklW,vs_Performance_PKwon,vs_Performance_PKcon,vs_Performance_OG
0,2,Arsenal,38,20,11,7,65,36,29,71,...,2,450,336,98,729,640,634,NaN,NaN,3
1,20,Aston Villa,38,3,8,27,27,76,-49,17,...,0,377,410,83,800,638,612,NaN,NaN,1
2,16,Bournemouth,38,11,9,18,45,67,-22,42,...,2,404,348,83,681,690,637,NaN,NaN,2
3,10,Chelsea,38,12,14,12,59,53,6,50,...,2,499,388,64,716,678,626,NaN,NaN,4
4,15,Crystal Palace,38,11,9,18,39,51,-12,42,...,2,448,458,47,791,523,651,NaN,NaN,1
5,11,Everton,38,11,14,13,59,55,4,47,...,0,426,299,70,871,686,598,NaN,NaN,3
6,1,Leicester City,38,23,12,3,68,36,32,81,...,0,374,390,71,1003,604,580,NaN,NaN,0
7,8,Liverpool,38,16,12,10,63,50,13,60,...,1,397,409,62,635,748,665,NaN,NaN,1
8,4,Manchester City,38,19,9,10,71,41,30,66,...,0,402,395,91,624,707,627,NaN,NaN,0
9,5,Manchester Utd,38,19,9,10,49,35,14,66,...,1,375,454,56,585,775,681,NaN,NaN,3


In [60]:
# Project overview — data aggregation for EPL stats (2015-16 to current)

# Official Premier League 3-letter team acronyms (use these for “team” column)
# Example mapping (not exhaustive; expand as needed for all clubs):
# ARS — Arsenal  
# AVL — Aston Villa  
# BOU — Bournemouth  
# BRE — Brentford  
# BRI — Brighton & Hove Albion  
# BUR — Burnley  
# CHE — Chelsea  
# CRY — Crystal Palace  
# EVE — Everton  
# FUL — Fulham  
# LIV — Liverpool  
# MCI — Manchester City  
# MUN — Manchester United  
# NEW — Newcastle United  
# NFO — Nottingham Forest  
# TOT — Tottenham Hotspur  
# WHU — West Ham United  
# WOL — Wolverhampton Wanderers  
# (If new/promoted clubs appear in any season, add their acronym accordingly.)

# Each data source / contributor has a dedicated function producing a dictionary:  
#   key = season (e.g. "2024-25")  
#   value = pandas.DataFrame with first column "team" (acronym), then one column per stat.

# Missing or unavailable stat values must be filled with the string "NONE" (unique placeholder).

# Output: export your season-by-season DataFrame dictionary to .csv -- one .csv with different tabs for one year (search up how to do this)

# Functions / authors:

# INJE: aggregate_understat()  
#   – uses scrape_understat()  
#   – returns dict[str, DataFrame] for each season  

# ALEX: aggregate_fbref()  
#   – uses scrape_fbref()  
#   – same output format  

# SHIVANTH: aggregate_sofascore() (or similar site for player metrics -- refer to GDoc for guidance)  
#   – gather derived stats, same output format  
#   - additionally beneficial if any site offers more coaching metrics of some sort (tactics, formations, etc) -- how to quantify that?

# Later:  
#   – When combining across sources, rely on “team” column (acronym) to align / merge data  -- Soham will do this  
#   – This yields a season and matches df per season
#     - Add selected metrics (e.g., xG, xA, PPDA, xGA, etc.) — note that for most data, these are **end-of-season aggregates**, so they are static and do not reflect in-season fluctuations.
#     - Compute contextual metrics when possible:
#         - Is_Home: 1 if home team, 0 if away  
#         - Last Five Game Points: sum of points over previous 5 matches for the team  
#         - Head-to-Head Points (last N): average points vs this opponent in last N meetings  
#         - Fill with "NONE" where insufficient historical matches exist (e.g., first 5 games of season)  

#   – Format the matches df consistently across all seasons  
#   – Add temporal weighting for features if desired (2015-16 = -10, 2016-17 = -9, ..., 2025-26 = 0)  
#   – For model training:  
#       - Use cross-validation with a rolling season scheme (e.g., train on 3 consecutive seasons, test on the next one)  
#       - Include all matches from matchday 1 of 2015-16 to last matchday in 2025-26  
#       - Features: team static metrics (FBref + Understat) + contextual features (last N matches, home/away, H2H, etc.)  
#       - Targets: win/draw/loss probabilities and goals scored/conceded (can model using LightGBM, XGBoost, or Poisson regression variants)  

# Example output (after one function):  
# {  
#   "2024-25": DataFrame with columns ["team", "xG", "xA", ...],  
#   "2023-24": DataFrame with same structure, ...  
# }  

In [ ]:
# --- INJE'S AGGREGATOR CODE --- #
# shared settings / helpers

TEAM_CODES = {
    "Arsenal": "ARS",
    "Aston Villa": "AVL",
    "Blackburn Rovers": "BLA",
    "Bolton Wanderers": "BOL",
    "Bournemouth": "BOU",
    "Brentford": "BRE",
    "Brighton": "BRI",
    "Brighton & Hove Albion": "BRI",
    "Burnley": "BUR",
    "Cardiff": "CAR",
    "Cardiff City": "CAR",
    "Chelsea": "CHE",
    "Crystal Palace": "CRY",
    "Derby County": "DER",
    "Everton": "EVE",
    "Fulham": "FUL",
    "Huddersfield": "HUD",
    "Huddersfield Town": "HUD",
    "Hull": "HUL",
    "Hull City": "HUL",
    "Ipswich": "IPS",
    "Ipswich Town": "IPS",
    "Leeds": "LEE",
    "Leeds United": "LEE",
    "Leicester": "LEI",
    "Leicester City": "LEI",
    "Liverpool": "LIV",
    "Luton": "LUT",
    "Luton Town": "LUT",
    "Manchester City": "MCI",
    "Manchester United": "MUN",
    "Middlesbrough": "MID",
    "Newcastle United": "NEW",
    "Norwich": "NOR",
    "Norwich City": "NOR",
    "Nottingham Forest": "NFO",
    "QPR": "QPR",
    "Queens Park Rangers": "QPR",
    "Reading": "RDG",
    "Sheffield": "SHU",
    "Sheffield United": "SHU",
    "Southampton": "SOU",
    "Stoke": "STK",
    "Stoke City": "STK",
    "Sunderland": "SUN",
    "Swansea": "SWA",
    "Swansea City": "SWA",
    "Tottenham": "TOT",
    "Tottenham Hotspur": "TOT",
    "Watford": "WAT",
    "West Bromwich Albion": "WBA",
    "West Ham": "WHU",
    "West Ham United": "WHU",
    "Wolverhampton Wanderers": "WOL",
}

MISSING_VALUE = "NONE"
SEASONS = list(range(2015, 2026))  # league year (final year of the season)

def to_team_code(name: str) -> str:
    """Convert Understat club names to 3-letter EPL acronyms."""
    if not isinstance(name, str):
        return MISSING_VALUE
    return TEAM_CODES.get(name.strip(), MISSING_VALUE)


def aggregate_understat(seasons=SEASONS):
    season_dfs = {}

    for year in seasons:
        season_label = f"{year}-{(year + 1) % 100:02d}"
        print(f"[start] {season_label} — launching scrape_understat")
        try:
            season_df, players_df, matches_df = scrape_understat("EPL", year)
            print(f"[done] {season_label} — season rows={len(season_df)}, players={len(players_df)}, matches={len(matches_df)}")
        except Exception as exc:
            print(f"[error] {season_label}: {exc}")
            season_dfs[season_label] = pd.DataFrame()
            continue

        if season_df.empty:
            print(f"[skip] {season_label} season table empty")
            season_dfs[season_label] = pd.DataFrame()
            continue

        df = season_df.copy()
        print(f"[debug] {season_label} season columns: {df.columns.tolist()}")

        name_col = next((c for c in ("Team", "Squad", "Club") if c in df.columns), None)
        if name_col is None:
            print(f"[error] {season_label} team column not found; columns={df.columns.tolist()}")
            season_dfs[season_label] = pd.DataFrame()
            continue

        df.rename(columns={name_col: "team"}, inplace=True)
        df = df[df["team"].str.lower() != "team"]
        df["team"] = df["team"].apply(to_team_code)

        # normalize xG / xGA / xPTS (first numeric token only)
        for metric in ("xG", "xGA", "xPTS"):
            if metric in df.columns:
                df[metric] = (
                    df[metric].astype(str)
                    .str.replace("+", " ", regex=False)
                    .str.replace("-", " ", regex=False)
                    .str.split()
                    .str[0]
                    .replace("", MISSING_VALUE)
                )

        value_cols = [c for c in df.columns if c != "team"]
        for col in value_cols:
            df[col] = pd.to_numeric(df[col], errors="ignore").replace("", MISSING_VALUE)

        df = df.fillna(MISSING_VALUE)
        season_dfs[season_label] = df
        print(f"[store] {season_label} dataframe shape {df.shape}")

    print(f"[summary] seasons collected: {list(season_dfs.keys())}")
    return season_dfs

# run aggregate_understat and inspect output

# Choose which seasons to run (adjust as needed)
seasons_to_run = list(range(2015, 2026))  # or SEASONS for full range

understat_data = aggregate_understat(seasons_to_run)

# peek at the first available season
if understat_data:
    first_key = next(iter(understat_data))
    print(f"Previewing season {first_key}")
    display(understat_data[first_key])
else:
    print("No data collected.")

# export understat dict to Excel (one sheet per season)

from pathlib import Path

output_path = Path("understat_by_season.xlsx")
if not understat_data:
    print("No Understat data to export.")
else:
    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        for season_key, df in understat_data.items():
            if df.empty:
                continue
            df.to_excel(writer, sheet_name=season_key.replace(":", "_"), index=False)
    print(f"Saved Understat output to {output_path.resolve()}")

list(understat_data.keys())